In [0]:
print("Smart Fraud Detection - Bronze Layer")
print("Notebook is working!")

path,name,size,modificationTime
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/accounts.csv,accounts.csv,21976,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/fraud_watchlist.csv,fraud_watchlist.csv,1576,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/transactions.csv,transactions.csv,963914,1786201673000


Step 2 — Read the CSV


In [0]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



%md
Step 1 — Check what file is inside Raw

In [0]:
raw_path = "/Volumes/smart_fraud_databricks/default/raw_data"

display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/accounts.csv,accounts.csv,21976,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/fraud_watchlist.csv,fraud_watchlist.csv,1576,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/transactions.csv,transactions.csv,963914,1786201673000


Step 2 — Read the CSV

In [0]:

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(raw_path)

display(df_raw)

txn_id,account_id,txn_date,amount,merchant
TXN009314,ACC0332,2025-01-10,107549.32,Swiggy
TXN013000,ACC0412,2025-03-17,22297.68,Zomato
TXN018018,ACC0486,2025-05-10,115208.63,Zomato
TXN013757,ACC0443,2025-07-24,45099.7,ATM_Withdrawal
TXN016494,ACC0062,2025-02-25,70884.04,Swiggy
TXN005278,ACC0012,2025-02-26,29630.42,Unknown_POS
TXN001327,ACC0009,2025-05-03,100371.89,Swiggy
TXN015333,ACC0096,2025-04-14,142083.52,Uber
TXN012891,ACC0307,2025-01-11,5800.25,Flipkart
TXN017015,ACC0170,2025-02-02,58011.24,Swiggy


Step 3 — Check the data

In [0]:
print("Rows:", df_raw.count())
print("Columns:", len(df_raw.columns))

df_raw.printSchema()

Rows: 20562
Columns: 5
root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- merchant: string (nullable = true)



Step 4 — Create the Bronze Delta table

In [0]:
df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("smart_fraud_databricks.default.bronze_transactions")

In [0]:
bronze_path = "/mnt/smart_fraud_databricks/default/bronze_data"

In [0]:
raw_path = "/Volumes/smart_fraud_databricks/default/raw_data"

display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/accounts.csv,accounts.csv,21976,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/fraud_watchlist.csv,fraud_watchlist.csv,1576,1786201671000
dbfs:/Volumes/smart_fraud_databricks/default/raw_data/transactions.csv,transactions.csv,963914,1786201673000


Read all 3 Raw CSV files

In [0]:
raw_path = "/Volumes/smart_fraud_databricks/default/raw_data"

accounts_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/accounts.csv")

transactions_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/transactions.csv")

watchlist_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/fraud_watchlist.csv")

Check the datasets

In [0]:
print("Accounts:", accounts_df.count())
print("Transactions:", transactions_df.count())
print("Watchlist:", watchlist_df.count())

print("\nAccounts Schema:")
accounts_df.printSchema()

print("\nTransactions Schema:")
transactions_df.printSchema()

print("\nWatchlist Schema:")
watchlist_df.printSchema()

Accounts: 505
Transactions: 20011
Watchlist: 46

Accounts Schema:
root
 |-- account_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- credit_limit: integer (nullable = true)
 |-- branch: string (nullable = true)


Transactions Schema:
root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- merchant: string (nullable = true)


Watchlist Schema:
root
 |-- account_id: string (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- flagged_date: string (nullable = true)



View the data

In [0]:
display(accounts_df.limit(10))

account_id,customer_name,account_type,credit_limit,branch
ACC0059,Customer_59,Salary,500000,Pune
ACC0139,Customer_139,Current,100000,Delhi
ACC0182,Customer_182,Salary,200000,Pune
ACC0302,Customer_302,Savings,500000,Mumbai
ACC0254,Customer_254,Savings,200000,Delhi
ACC0255,Customer_255,Current,100000,Bangalore
ACC0007,Customer_7,Salary,50000,Pune
ACC0325,Customer_325,Current,500000,Bangalore
ACC0105,Customer_105,Current,50000,Mumbai
ACC0221,Customer_221,Salary,100000,Mumbai


In [0]:
display(transactions_df.limit(10))

txn_id,account_id,txn_date,amount,merchant
TXN009314,ACC0332,2025-01-10,107549.32,Swiggy
TXN013000,ACC0412,2025-03-17,22297.68,Zomato
TXN018018,ACC0486,2025-05-10,115208.63,Zomato
TXN013757,ACC0443,2025-07-24,45099.7,ATM_Withdrawal
TXN016494,ACC0062,2025-02-25,70884.04,Swiggy
TXN005278,ACC0012,2025-02-26,29630.42,Unknown_POS
TXN001327,ACC0009,2025-05-03,100371.89,Swiggy
TXN015333,ACC0096,2025-04-14,142083.52,Uber
TXN012891,ACC0307,2025-01-11,5800.25,Flipkart
TXN017015,ACC0170,2025-02-02,58011.24,Swiggy


In [0]:
display(watchlist_df.limit(10))

account_id,fraud_type,flagged_date
ACC0116,Account Takeover,2025-05-26
ACC0300,Card Skimming,10-Feb-2025
ACC0280,Account Takeover,2025-03-03
ACC0464,Account Takeover,2025-07-14
ACC0072,Account Takeover,2025-05-17
ACC0178,Card Skimming,2025-02-17
ACC0268,Card Skimming,2025-07-11
ACC0085,Card Skimming,2025-07-11
ACC0013,Card Skimming,2025-05-19
ACC0208,Card Skimming,2025-06-11


Create Bronze Delta tables

In [0]:
bronze_path = "/Volumes/smart_fraud_databricks/default/raw_data/bronze"

# Accounts
accounts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/accounts")

# Transactions
transactions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/transactions")

# Fraud Watchlist
watchlist_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/fraud_watchlist")

print("Bronze Delta layer created successfully!")

Bronze Delta layer created successfully!


Verify Bronze

In [0]:
bronze_accounts = spark.read.format("delta").load(
    f"{bronze_path}/accounts"
)

bronze_transactions = spark.read.format("delta").load(
    f"{bronze_path}/transactions"
)

bronze_watchlist = spark.read.format("delta").load(
    f"{bronze_path}/fraud_watchlist"
)

print("Bronze Accounts:", bronze_accounts.count())
print("Bronze Transactions:", bronze_transactions.count())
print("Bronze Watchlist:", bronze_watchlist.count())

Bronze Accounts: 505
Bronze Transactions: 20011
Bronze Watchlist: 46


View Bronze transactions

In [0]:
display(bronze_transactions.limit(20))

txn_id,account_id,txn_date,amount,merchant
TXN009314,ACC0332,2025-01-10,107549.32,Swiggy
TXN013000,ACC0412,2025-03-17,22297.68,Zomato
TXN018018,ACC0486,2025-05-10,115208.63,Zomato
TXN013757,ACC0443,2025-07-24,45099.7,ATM_Withdrawal
TXN016494,ACC0062,2025-02-25,70884.04,Swiggy
TXN005278,ACC0012,2025-02-26,29630.42,Unknown_POS
TXN001327,ACC0009,2025-05-03,100371.89,Swiggy
TXN015333,ACC0096,2025-04-14,142083.52,Uber
TXN012891,ACC0307,2025-01-11,5800.25,Flipkart
TXN017015,ACC0170,2025-02-02,58011.24,Swiggy
